# Hisaab Aunty: learn finance, Indian tax and banking, English and Hinglish

**Press Runtime > Run all.** Safe to re-run any time: if Colab disconnects, run it again and it carries on from the last saved step (progress is saved to your Google Drive every 50 steps).

| Cell | What it does |
|---|---|
| 1 | Connects Drive, picks the model to continue from, sets the folders |
| 2 | Installs the tools and downloads our code and our Hinglish examples from GitHub |
| 3 | Collects the training data **once** (public finance, English, Hindi and Hinglish sets, real company statements, our own examples) and saves it in Drive |
| 4 | Trains. Saves progress every 50 steps. Writes the finished model to Drive |
| 5 | Credits file, then the model answers a few questions so you can read them |

**What the model learns from:** full written answers (explanations, worked calculations, conversations), not only labels. The training loss is counted on the whole answer.

**Before you start:** Runtime > Change runtime type > T4 GPU. Your model folder must be in Google Drive (My Drive/Sent_model_hinglish or My Drive/Sent_model).

Saved in Drive next to your model: Hisaab_work/mix_finance (training data), Hisaab_work/checkpoints_finance (progress), Sent_model_finance (the finished model).

In [ ]:
# 1. Setup: Drive, the model to continue from, and where everything is saved
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, re, shutil

# continue from the Hinglish model if it is in Drive, otherwise from the original model
BASE = next((p for p in ['/content/drive/MyDrive/Sent_model_hinglish', '/content/drive/MyDrive/Sent_model', '/content/drive/MyDrive/Hisaab/Sent_model'] if os.path.exists(p + '/model.safetensors')), None)
assert BASE, 'Model folder not found. Put it in Google Drive as My Drive/Sent_model_hinglish or My Drive/Sent_model'
PARENT = os.path.dirname(BASE)
MIX = PARENT + '/Hisaab_work/mix_finance'              # training data (collected once)
CKPT = PARENT + '/Hisaab_work/checkpoints_finance'     # training progress (saved every 50 steps)
OUT = PARENT + '/Sent_model_finance'                   # the finished model
EPOCHS = 1                                             # passes over the data. The data is large, so 1 pass is plenty
RANK = 32                                              # size of the learning add-on (bigger = room for more knowledge)
SEC_EMAIL = 'research@example.com'                     # the US SEC asks downloaders to give an email; put yours if you like

missing = [f for f in ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json'] if not os.path.exists(BASE + '/' + f)]
assert not missing, 'MISSING in ' + BASE + ': ' + ', '.join(missing)
print('continuing from:', BASE, '\nsaves to:', PARENT)

In [ ]:
# 2. Install the tools and download our code and our Hinglish examples from GitHub
!pip install -q -U transformers peft trl datasets accelerate yfinance
!pip uninstall -y -q torchao
!rm -rf /content/repo && git clone -q https://github.com/sohail807/hisaab_aunty /content/repo
!ls /content/repo/prototype/training/data

In [ ]:
# 3. Collect the training data. Only the first time: it is saved in Drive and reused after that.
R = '/content/repo/prototype/training/data'

if os.path.exists(MIX + '/train.jsonl'):
    print('Training data already saved in Drive, reusing it:', MIX)
    print(open(MIX + '/summary.txt').read() if os.path.exists(MIX + '/summary.txt') else '')
else:
    from itertools import islice
    from datasets import load_dataset
    random.seed(7)
    SYS = 'You are a knowledgeable, careful assistant for money and finance. Answer clearly in the language the user writes in (English or Hinglish) and show your working for any calculation.'
    read = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
    counts = {}; train, val = [], []

    def add(group, name, pairs, system=SYS):
        rows = [{'id': name, 'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': q}, {'role': 'assistant', 'content': a}]} for q, a in pairs]
        hold = max(1, len(rows) // 100)
        val.extend(rows[:hold]); train.extend(rows[hold:]); counts[name] = (group, len(rows))
        print(f'  {group:16} {name:55} {len(rows):>6} examples')

    def texts(x):
        if isinstance(x, str): yield x
        elif isinstance(x, dict):
            for v in x.values(): yield from texts(v)
        elif isinstance(x, (list, tuple)):
            for v in x: yield from texts(v)

    def pair(r):   # find the question and the answer in whatever shape the dataset uses
        turns = r.get('messages') or r.get('conversations') or r.get('conversation')
        if isinstance(turns, list) and turns:
            q = a = ''
            for t in turns:
                if not isinstance(t, dict): continue
                role = str(t.get('role') or t.get('from') or '').lower(); text = str(t.get('content') or t.get('value') or '').strip()
                if role in ('user', 'human') and not q: q = text
                elif role in ('assistant', 'gpt', 'bot') and q and not a: a = text
            return q, a
        g = lambda *ks: next((str(r[k]).strip() for k in ks if r.get(k) not in (None, '') and str(r[k]).strip()), '')
        q = g('instruction', 'question', 'query', 'user', 'prompt'); extra = g('input', 'context')
        return (q + ('\n\n' + extra if extra else '')).strip(), g('output', 'response', 'answer', 'assistant', 'completion')

    def act_pair(r):   # the Income Tax Act, one section at a time: "what does section X say" -> the section text
        text = str(r.get('content') or '').strip(); title = str(r.get('section_title') or '').strip().rstrip('.')
        if len(text) < 150: return '', ''   # headings and one-line stubs teach nothing
        return f"What does Section {r.get('section_id')} of the {r.get('act_name') or 'Income Tax Act'} say? ({title})", text

    def rbi_pair(r):   # RBI banking regulation: the long explained answer if it fits, otherwise the short one
        q, a = str(r.get('enhanced_prompt') or '').strip(), str(r.get('enhanced_completion') or '').strip()
        if q and a and len(q) + len(a) <= 3500: return q, a
        return str(r.get('input') or '').strip(), str(r.get('output') or '').strip()

    KEEP = {'Priyanshu-24/adaption-indian-finance-qa': ['enhanced_prompt', 'enhanced_completion', 'input', 'output']}

    def fetch(hf_id, cap):   # stream a shuffled sample, so no huge downloads
        ds = load_dataset(hf_id, split='train', streaming=True)
        if hf_id in KEEP: ds = ds.select_columns(KEEP[hf_id])   # skip heavy columns we do not need (the RBI set carries a big vector on every row)
        return list(islice(ds.shuffle(seed=7, buffer_size=5000), cap * 2))

    print('Public datasets (anything that cannot be loaded is skipped and listed):')
    SOURCES = [   # (group, Hugging Face dataset, how many examples to take)
        ('finance', 'Josephgflowers/Finance-Instruct-500k', 14000), ('finance', 'gbharti/finance-alpaca', 8000), ('finance', 'FinGPT/fingpt-fiqa_qa', 5000), ('finance filings', 'virattt/financial-qa-10K', 7000),
        ('english', 'yahma/alpaca-cleaned', 6000), ('english', 'databricks/databricks-dolly-15k', 5000),
        ('hindi/hinglish', 'ravithejads/samvaad-hi-filtered', 4000), ('hindi/hinglish', 'FreedomIntelligence/alpaca-gpt4-hindi', 4000),
        ('hindi/hinglish', 'smangrul/hinglish_self_instruct_v0', 4000),
        ('india tax law', 'varshith7/income-tax-act-india', 9000, act_pair), ('india tax q&a', 'ligaments-dev/indian-income-tax-qa', 8000), ('india tax q&a', 'keerthan222/indian-tax-law-qa', 300),
        ('india banking (RBI)', 'Priyanshu-24/adaption-indian-finance-qa', 9000, rbi_pair),
    ]
    skipped = []
    for group, hf_id, cap, *own in SOURCES:
        try:
            pairs = []
            for r in fetch(hf_id, cap):
                q, a = (own[0] if own else pair)(r)
                if q and a and len(a) >= 15 and 40 <= len(q) + len(a) <= 3500: pairs.append((q, a))
                if len(pairs) >= cap: break
            if pairs: add(group, hf_id, pairs)
            else: skipped.append(hf_id + ' (no usable rows)')
        except Exception as e: skipped.append(hf_id + ' (' + repr(e)[:90] + ')')

    # Hinglish everyday conversations: used as real chat (line answers the line before) and as "do nothing" examples for the app
    SKIP = re.compile(r"(\d|\u20b9|\brs\b|rupee|rupay|paisa|paise|kharch|bacha|budget|hisaab|bill|udhaar|salary|kamai|bank|loan|emi|khata|price|amount|balance|account|limit|spend|spent|expense|save|saving|goal|streak|recap|undo|delete|cancel|transfer|payment|paid|\bpay\b|card|cash|upi|refund|charge|fee|tax|invest|share|stock|crypto|fund|insurance|\bhi\b|hello|namaste|kaise|kaisi|thank|shukriya|bye|good (morning|night)|aunty|http|www|@)", re.I)
    systems = [r['messages'][0]['content'] for r in read(R + '/train.jsonl')]       # the exact instruction the app sends
    neg = lambda t: {'id': 'real_neg', 'messages': [{'role': 'system', 'content': random.choice(systems)}, {'role': 'user', 'content': t}, {'role': 'assistant', 'content': '{"intent":"unknown"}'}]}
    try:
        hrows = list(islice(load_dataset('Abhishekcr448/Hinglish-Everyday-Conversations-1M', split='train', streaming=True), 30000))
        chat, lines_ok = [], set()
        for row in hrows:
            ls = [re.sub(r'^\s*(person\s*\d*|user|speaker\s*\d*|human|assistant|[AB])\s*[:\-]\s*', '', l, flags=re.I).strip() for s in texts(row) for l in s.split('\n')]
            ls = [l for l in ls if l]
            for a_, b_ in zip(ls, ls[1:]):
                if 3 <= len(a_.split()) <= 40 and 3 <= len(b_.split()) <= 60: chat.append((a_, b_))
            for l in ls:
                if 3 <= len(l.split()) <= 14 and re.fullmatch(r"[A-Za-z ,.'?!\-]+", l) and not SKIP.search(l): lines_ok.add(l)
        random.shuffle(chat); add('hinglish chat', 'Abhishekcr448/Hinglish-Everyday-Conversations-1M', chat[:8000])
        negs = sorted(lines_ok); random.shuffle(negs); train.extend(neg(t) for t in negs[:1500]); counts['Hinglish do-nothing lines'] = ('app', min(1500, len(negs)))
    except Exception as e: skipped.append('Abhishekcr448/Hinglish-Everyday-Conversations-1M (' + repr(e)[:90] + ')')
    for name in ['PolyAI/banking77', 'legacy-datasets/banking77']:
        try:
            bank = sorted({r['text'].strip() for r in load_dataset(name, split='train') if 3 <= len(r['text'].split()) <= 14}); random.shuffle(bank)
            train.extend(neg(t) for t in bank[:1000]); counts['Banking77 do-nothing lines'] = ('app', min(1000, len(bank))); break
        except Exception as e: last_err = name + ' (' + repr(e)[:90] + ')'
    else: skipped.append(last_err)

    # our own Hinglish examples from GitHub (what the app needs), kept in full so nothing already learned is lost
    ours_t, ours_v = read(R + '/train.jsonl'), read(R + '/val.jsonl'); train.extend(ours_t); val.extend(ours_v[:100]); counts['our GitHub Hinglish examples'] = ('app', len(ours_t))
    print(f'  {"app":16} {"our GitHub Hinglish examples":55} {len(ours_t):>6} examples')

    # real company statements (US: SEC filings, India: Yahoo Finance), answers computed in code from the real figures
    import subprocess
    FIN = PARENT + '/Hisaab_work/fin_data'
    if not os.path.exists(FIN + '/train.jsonl'):
        r = subprocess.run(['python', '/content/repo/prototype/training/fin/build_fin_data.py', '--out', FIN, '--india'], env=dict(os.environ, SEC_USER_AGENT='HisaabAunty Research ' + SEC_EMAIL), capture_output=True, text=True)
        print((r.stdout or '')[-400:], (r.stderr or '')[-300:])
    if os.path.exists(FIN + '/train.jsonl'):
        ft, fv, fx = read(FIN + '/train.jsonl'), read(FIN + '/val.jsonl'), read(FIN + '/test.jsonl')
        train.extend(ft + fx); val.extend(fv); counts['real company statements (SEC + India)'] = ('finance statements', len(ft) + len(fx) + len(fv))
        print(f'  {"finance statements":16} {"real company statements (SEC + India)":55} {len(ft) + len(fx) + len(fv):>6} examples')
    else: skipped.append('real company statements (could not be built, see the lines above)')

    random.shuffle(train); random.shuffle(val)
    os.makedirs(MIX, exist_ok=True)
    for name, rows in (('train', train), ('val', val)):
        open(MIX + '/' + name + '.jsonl', 'w', encoding='utf-8').write('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + '\n')
    summary = 'TOTAL train ' + str(len(train)) + ' | val ' + str(len(val)) + ('\nSKIPPED:\n  ' + '\n  '.join(skipped) if skipped else '\nnothing skipped')
    json.dump({'sources': counts}, open(MIX + '/sources.json', 'w'), indent=1); open(MIX + '/summary.txt', 'w').write(summary)
    print('\n' + summary + '\nsaved to Drive:', MIX)

## 4. Train
Saves progress to Drive every 50 steps. If Colab disconnects, press Run all again: cells 1 to 3 are quick and this cell continues from the last saved step. The data is large, so it can take several sessions; every session adds to the same saved progress. Keep this tab open and the laptop awake.

In [ ]:
# 4. Train. Continues from saved progress; writes the finished model straight to Drive.
!python /content/repo/prototype/training/train_lora.py --base "$BASE" --data "$MIX" --out "$OUT" --ckpt "$CKPT" --epochs $EPOCHS --rank $RANK

In [ ]:
# 5. Credits file, then the model answers a few questions so you can read them
sources = json.load(open(MIX + '/sources.json'))['sources'] if os.path.exists(MIX + '/sources.json') else {}
open(OUT + '/DATA_SOURCES.md', 'w').write('# Data used to train this model\n\n- Our own Hinglish examples: https://github.com/sohail807/hisaab_aunty (prototype/training)\n- Real company statements: SEC EDGAR (US) and Yahoo Finance (India), built by prototype/training/fin/build_fin_data.py\n' + ''.join('- ' + k + ': ' + str(v[1]) + ' examples (' + v[0] + ')\n' for k, v in sources.items()) + '\nCheck each dataset license before commercial use.\n')

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(OUT)
model = AutoModelForCausalLM.from_pretrained(OUT, dtype=torch.float16, device_map='auto')
SYS = 'You are a knowledgeable, careful assistant for money and finance. Answer clearly in the language the user writes in (English or Hinglish) and show your working for any calculation.'
for q in ['What is the difference between a balance sheet and a cash flow statement?',
          'A company has current assets of 150 crore and current liabilities of 100 crore. What is its current ratio and what does it mean?',
          'Explain free cash flow in simple words.',
          'EMI aur SIP me kya fark hota hai? Simple Hinglish me samjhao.',
          'Net income is 40 and operating cash flow is 25. Is the profit backed by cash?',
          'What does Section 80C of the Income Tax Act allow an individual to claim?',
          'TDS on professional fees under Section 194J kab lagta hai aur kitna?',
          'What does the RBI expect banks to do for KYC of customers?']:
    ids = tok(tok.apply_chat_template([{'role': 'system', 'content': SYS}, {'role': 'user', 'content': q}], tokenize=False, add_generation_prompt=True), return_tensors='pt').to(model.device)
    out = model.generate(**ids, max_new_tokens=220, do_sample=False)
    print('\nQ:', q, '\nA:', tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip())

## Done
The finished model is in Drive: Sent_model_finance (about 6 GB). Read the answers above. The next step on your PC is to download it and convert it to GGUF.